In [1]:
import numpy as np
import scanpy as sc
from scipy.sparse import issparse

# -----------------------------
# Config
# -----------------------------
PATH = "/work3/s252608/DL_project/data/processed/bulk_normalized_x_input.h5ad"
BATCH_SIZE = 512

# -----------------------------
# Open in backed mode (low RAM)
# -----------------------------
adata = sc.read_h5ad(PATH, backed="r")

n_obs, n_vars = adata.shape
print(f"Shape: {n_obs:,} samples × {n_vars:,} genes")

# -----------------------------
# Streaming variance computation
# Welford online algorithm
# -----------------------------
mean = np.zeros(n_vars, dtype=np.float64)
M2 = np.zeros(n_vars, dtype=np.float64)
count = 0

for start in range(0, n_obs, BATCH_SIZE):
    end = min(start + BATCH_SIZE, n_obs)

    X_batch = adata.X[start:end]

    if issparse(X_batch):
        X_batch = X_batch.toarray()

    X_batch = np.asarray(X_batch, dtype=np.float64)

    for x in X_batch:
        count += 1
        delta = x - mean
        mean += delta / count
        delta2 = x - mean
        M2 += delta * delta2

    print(f"{end:,}/{n_obs:,}", end="\r")

variance = M2 / (count - 1)
std = np.sqrt(variance)

print("\nDone.")

# -----------------------------
# Summary statistics
# -----------------------------
print("\nVariance summary:")
print(f"Mean variance:   {variance.mean():.4f}")
print(f"Median variance: {np.median(variance):.4f}")
print(f"Min variance:    {variance.min():.4f}")
print(f"Max variance:    {variance.max():.4f}")

print("\nStd summary:")
print(f"Mean std:        {std.mean():.4f}")
print(f"Median std:      {np.median(std):.4f}")

Shape: 19,882 samples × 2,204 genes
19,882/19,882
Done.

Variance summary:
Mean variance:   0.2720
Median variance: 0.1987
Min variance:    0.0004
Max variance:    4.1202

Std summary:
Mean std:        0.4743
Median std:      0.4458
